In [1]:
# FIFA World Cup 2026 Prediction Engine
## Notebook 3: Model Training and Evaluation

# This notebook:
# - Loads engineered features
# - Splits data into train and test sets
# - Trains Random Forest and XGBoost models
# - Evaluates predictive performance
# - Selects the final model for tournament simulation

In [2]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import xgboost as xgb

PROCESSED_DIR = "data/processed"
MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

In [3]:
## Load Engineered Dataset

In [4]:
df = pd.read_csv(f"{PROCESSED_DIR}/training_dataset_features.csv")
print("Loaded shape:", df.shape)

identifier_cols = ["year", "home_team", "away_team"]
target_col = "match_result"

engineered_features = [c for c in df.columns if c not in identifier_cols + [target_col]]

X = df[engineered_features].copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print(f"\n{len(engineered_features)} features:")
for f in engineered_features:
    print(" ", f)

Loaded shape: (384, 31)
X shape: (384, 27)
y shape: (384,)

27 features:
  draws_last_4y_difference
  fifa_points_pre_tournament_difference
  fifa_rank_pre_tournament_difference
  finalist_difference
  finals_before_difference
  goals_received_last_4y_difference
  goals_scored_last_4y_difference
  groups_passed_before_difference
  is_host_difference
  losses_last_4y_difference
  quarter_finalist_difference
  quarterfinals_before_difference
  round16_before_difference
  semi_finalist_difference
  semifinals_before_difference
  squad_avg_age_difference
  squad_total_market_value_eur_difference
  winner_difference
  wins_last_4y_difference
  world_cup_participations_before_difference
  world_cup_titles_before_difference
  fifa_points_pre_tournament_ratio
  goals_scored_last_4y_ratio
  goals_received_last_4y_ratio
  wins_last_4y_ratio
  squad_avg_age_ratio
  host_difference


In [5]:
## Target Variable Definition

In [6]:
# These represent THIS tournament's final outcome for each team 
outcome_leakage_features = [
    "winner_difference",
    "finalist_difference",
    "semi_finalist_difference",
    "quarter_finalist_difference",
]

present = [c for c in outcome_leakage_features if c in X.columns]
print("Outcome-leakage features found:", present)

if present:
    print("\nChecking how strongly these align with match_result:")
    check_df = X[present].copy()
    check_df["match_result"] = y.values

    for col in present:
        sign = check_df[col].apply(lambda v: "diff > 0" if v > 0 else ("diff < 0" if v < 0 else "diff = 0"))
        print(f"\n{col}:")
        print(pd.crosstab(sign, check_df["match_result"], normalize="index").round(2))

    X = X.drop(columns=present)
    print(f"\nDropped {len(present)} leakage columns.")
    print(f"New X shape: {X.shape}")

Outcome-leakage features found: ['winner_difference', 'finalist_difference', 'semi_finalist_difference', 'quarter_finalist_difference']

Checking how strongly these align with match_result:

winner_difference:
match_result       Away Win  Draw  Home Win
winner_difference                          
diff < 0               0.88  0.12      0.00
diff = 0               0.34  0.24      0.42
diff > 0               0.08  0.15      0.77

finalist_difference:
match_result         Away Win  Draw  Home Win
finalist_difference                          
diff < 0                 0.82  0.14      0.04
diff = 0                 0.34  0.25      0.41
diff > 0                 0.05  0.14      0.82

semi_finalist_difference:
match_result              Away Win  Draw  Home Win
semi_finalist_difference                          
diff < 0                      0.80  0.18      0.02
diff = 0                      0.34  0.24      0.42
diff > 0                      0.04  0.23      0.73

quarter_finalist_difference:
match_

In [7]:
team_features_2026 = pd.read_csv(f"{PROCESSED_DIR}/team_features_2026.csv")
print("team_features_2026.csv columns:")
print(list(team_features_2026.columns))


def get_base_name(feature_name):
    if feature_name.endswith("_difference"):
        return feature_name[:-len("_difference")]
    if feature_name.endswith("_ratio"):
        return feature_name[:-len("_ratio")]
    return feature_name


bases_needed = sorted(set(get_base_name(f) for f in X.columns))
team_2026_cols = set(team_features_2026.columns)

print(f"\nChecking {len(bases_needed)} base columns needed to build 2026 features:")
missing_bases = []
for b in bases_needed:
    found = b in team_2026_cols
    status = "OK" if found else "MISSING"
    print(f"  {b}: {status}")
    if not found:
        missing_bases.append(b)

print(f"\n{len(missing_bases)} base column(s) not directly available in team_features_2026.csv:")
print(missing_bases)

team_features_2026.csv columns:
['team', 'squad_size', 'average_age', 'average_height', 'total_caps', 'average_caps', 'total_goals', 'average_goals', 'number_of_goalkeepers', 'number_of_defenders', 'number_of_midfielders', 'number_of_forwards', 'team_code', 'association', 'rank', 'previous_rank', 'points', 'previous_points', 'rated_matches', 'manager_name', 'manager_nationality', 'appointment_year', 'preferred_formation', 'style_description']

Checking 18 base columns needed to build 2026 features:
  draws_last_4y: MISSING
  fifa_points_pre_tournament: MISSING
  fifa_rank_pre_tournament: MISSING
  finals_before: MISSING
  goals_received_last_4y: MISSING
  goals_scored_last_4y: MISSING
  groups_passed_before: MISSING
  host: MISSING
  is_host: MISSING
  losses_last_4y: MISSING
  quarterfinals_before: MISSING
  round16_before: MISSING
  semifinals_before: MISSING
  squad_avg_age: MISSING
  squad_total_market_value_eur: MISSING
  wins_last_4y: MISSING
  world_cup_participations_before: MI

In [8]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label mapping:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls} -> {i}")

joblib.dump(label_encoder, f"{MODELS_DIR}/label_encoder.pkl")
print(f"\nSaved to {MODELS_DIR}/label_encoder.pkl")

Label mapping:
  Away Win -> 0
  Draw -> 1
  Home Win -> 2

Saved to models/label_encoder.pkl


In [9]:
## Train-Test Split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)


def class_distribution(y_arr, name):
    counts = pd.Series(y_arr).value_counts().sort_index()
    counts.index = label_encoder.inverse_transform(counts.index)
    pct = (counts / counts.sum() * 100).round(1)
    summary = pd.DataFrame({"count": counts, "%": pct})
    print(f"\n{name} distribution:")
    print(summary)


class_distribution(y_train, "Train")
class_distribution(y_test, "Test")

Train shape: (307, 23)
Test shape:  (77, 23)

Train distribution:
          count     %
Away Win    105  34.2
Draw         70  22.8
Home Win    132  43.0

Test distribution:
          count     %
Away Win     26  33.8
Draw         18  23.4
Home Win     33  42.9


In [11]:
## Random Forest Model

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("Random Forest trained.")
print(f"  n_estimators = {rf_model.n_estimators}")
print(f"  max_depth = {rf_model.max_depth}")
print(f"  min_samples_leaf = {rf_model.min_samples_leaf}")

Random Forest trained.
  n_estimators = 300
  max_depth = 6
  min_samples_leaf = 5


In [13]:
rf_preds = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_preds)
rf_precision = precision_score(y_test, rf_preds, average="macro")
rf_recall = recall_score(y_test, rf_preds, average="macro")
rf_f1 = f1_score(y_test, rf_preds, average="macro")

print(f"Accuracy:           {rf_accuracy:.3f}")
print(f"Precision (macro):  {rf_precision:.3f}")
print(f"Recall (macro):     {rf_recall:.3f}")
print(f"F1 (macro):         {rf_f1:.3f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, rf_preds)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {c}" for c in label_encoder.classes_],
    columns=[f"Pred: {c}" for c in label_encoder.classes_]
)
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test, rf_preds, target_names=label_encoder.classes_))

Accuracy:           0.532
Precision (macro):  0.455
Recall (macro):     0.469
F1 (macro):         0.454

Confusion Matrix:
                  Pred: Away Win  Pred: Draw  Pred: Home Win
Actual: Away Win              14           3               9
Actual: Draw                   7           2               9
Actual: Home Win               3           5              25

Classification Report:
              precision    recall  f1-score   support

    Away Win       0.58      0.54      0.56        26
        Draw       0.20      0.11      0.14        18
    Home Win       0.58      0.76      0.66        33

    accuracy                           0.53        77
   macro avg       0.45      0.47      0.45        77
weighted avg       0.49      0.53      0.50        77



In [14]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print("Feature importance ranking:")
print(importances.round(4))

print("\nTop 10 features:")
print(importances.head(10))

Feature importance ranking:
squad_total_market_value_eur_difference       0.1072
fifa_rank_pre_tournament_difference           0.0917
fifa_points_pre_tournament_difference         0.0749
fifa_points_pre_tournament_ratio              0.0707
round16_before_difference                     0.0568
goals_received_last_4y_ratio                  0.0498
wins_last_4y_ratio                            0.0475
semifinals_before_difference                  0.0461
groups_passed_before_difference               0.0441
goals_scored_last_4y_difference               0.0437
goals_received_last_4y_difference             0.0421
squad_avg_age_ratio                           0.0418
world_cup_participations_before_difference    0.0412
goals_scored_last_4y_ratio                    0.0376
losses_last_4y_difference                     0.0369
squad_avg_age_difference                      0.0362
quarterfinals_before_difference               0.0356
draws_last_4y_difference                      0.0348
wins_last_4y_diffe

In [15]:
## XGBoost Model

In [16]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train, y_train)
print("XGBoost trained.")
print(f"  n_estimators = {xgb_model.n_estimators}")
print(f"  max_depth = {xgb_model.max_depth}")
print(f"  learning_rate = {xgb_model.learning_rate}")

XGBoost trained.
  n_estimators = 300
  max_depth = 3
  learning_rate = 0.05


In [17]:
xgb_preds = xgb_model.predict(X_test)

xgb_accuracy = accuracy_score(y_test, xgb_preds)
xgb_precision = precision_score(y_test, xgb_preds, average="macro")
xgb_recall = recall_score(y_test, xgb_preds, average="macro")
xgb_f1 = f1_score(y_test, xgb_preds, average="macro")

print(f"Accuracy:           {xgb_accuracy:.3f}")
print(f"Precision (macro):  {xgb_precision:.3f}")
print(f"Recall (macro):     {xgb_recall:.3f}")
print(f"F1 (macro):         {xgb_f1:.3f}")

print("\nConfusion Matrix:")
cm_xgb = confusion_matrix(y_test, xgb_preds)
cm_xgb_df = pd.DataFrame(
    cm_xgb,
    index=[f"Actual: {c}" for c in label_encoder.classes_],
    columns=[f"Pred: {c}" for c in label_encoder.classes_]
)
print(cm_xgb_df)

print("\nClassification Report:")
print(classification_report(y_test, xgb_preds, target_names=label_encoder.classes_))

Accuracy:           0.468
Precision (macro):  0.377
Recall (macro):     0.399
F1 (macro):         0.377

Confusion Matrix:
                  Pred: Away Win  Pred: Draw  Pred: Home Win
Actual: Away Win              10           4              12
Actual: Draw                   7           1              10
Actual: Home Win               3           5              25

Classification Report:
              precision    recall  f1-score   support

    Away Win       0.50      0.38      0.43        26
        Draw       0.10      0.06      0.07        18
    Home Win       0.53      0.76      0.62        33

    accuracy                           0.47        77
   macro avg       0.38      0.40      0.38        77
weighted avg       0.42      0.47      0.43        77



In [18]:
## Model Comparison

In [19]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "Accuracy": [rf_accuracy, xgb_accuracy],
    "Precision (macro)": [rf_precision, xgb_precision],
    "Recall (macro)": [rf_recall, xgb_recall],
    "F1 (macro)": [rf_f1, xgb_f1],
})

print(comparison.round(3).to_string(index=False))

        Model  Accuracy  Precision (macro)  Recall (macro)  F1 (macro)
Random Forest     0.532              0.455           0.469       0.454
      XGBoost     0.468              0.377           0.399       0.377


In [20]:
## Final Model Selection

In [21]:
if xgb_f1 >= rf_f1:
    best_model = xgb_model
    best_model_name = "XGBoost"
    best_f1 = xgb_f1
else:
    best_model = rf_model
    best_model_name = "Random Forest"
    best_f1 = rf_f1

print(f"Selected model: {best_model_name}")
print(f"  (macro F1 = {best_f1:.3f})")

joblib.dump(best_model, f"{MODELS_DIR}/best_model.pkl")
print(f"\nSaved best model to: {MODELS_DIR}/best_model.pkl")
print(f"Label encoder already saved to: {MODELS_DIR}/label_encoder.pkl")

print(f"\nFinal feature set used ({len(X.columns)} features) - required for Notebook 04:")
for c in X.columns:
    print(" ", c)

Selected model: Random Forest
  (macro F1 = 0.454)

Saved best model to: models/best_model.pkl
Label encoder already saved to: models/label_encoder.pkl

Final feature set used (23 features) - required for Notebook 04:
  draws_last_4y_difference
  fifa_points_pre_tournament_difference
  fifa_rank_pre_tournament_difference
  finals_before_difference
  goals_received_last_4y_difference
  goals_scored_last_4y_difference
  groups_passed_before_difference
  is_host_difference
  losses_last_4y_difference
  quarterfinals_before_difference
  round16_before_difference
  semifinals_before_difference
  squad_avg_age_difference
  squad_total_market_value_eur_difference
  wins_last_4y_difference
  world_cup_participations_before_difference
  world_cup_titles_before_difference
  fifa_points_pre_tournament_ratio
  goals_scored_last_4y_ratio
  goals_received_last_4y_ratio
  wins_last_4y_ratio
  squad_avg_age_ratio
  host_difference
